# Week 2: 概率模拟

## 学习目标

1. 理解概率的基本概念：样本空间、事件、概率分布
2. 掌握 Monte Carlo 模拟方法
3. 学习大数定律和中心极限定理的直观理解
4. 用模拟方法解决实际问题

## 1. 概率基础回顾

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# 设置绘图参数
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("概率模拟工具已加载")

### 1.1 样本空间与事件

- **样本空间 (Ω)**：所有可能结果的集合
- **事件 (A)**：样本空间的子集
- **概率 P(A)**：事件 A 发生的可能性，取值 [0, 1]

In [ ]:
# 示例：掷骰子
# 样本空间: {1, 2, 3, 4, 5, 6}
# 事件 A: 偶数 {2, 4, 6}
# P(A) = 3/6 = 0.5

def roll_die(n_trials):
    """模拟掷骰子 n 次"""
    return np.random.randint(1, 7, size=n_trials)

# 模拟 10000 次掷骰子
n = 10000
results = roll_die(n)

# 计算偶数的频率
even_count = np.sum(results % 2 == 0)
empirical_prob = even_count / n

print(f"理论概率: 0.5")
print(f"模拟结果: {empirical_prob:.4f}")
print(f"误差: {abs(empirical_prob - 0.5):.4f}")

## 2. Monte Carlo 模拟

### 2.1 估计圆周率 π

**方法**：在单位正方形内随机撒点，计算落在内切圆中的比例。

In [ ]:
def estimate_pi(n_points):
    """用 Monte Carlo 方法估计 π"""
    # 随机生成点 (x, y) ∈ [0, 1] × [0, 1]
    x = np.random.random(n_points)
    y = np.random.random(n_points)
    
    # 计算到圆心的距离
    distance = np.sqrt(x**2 + y**2)
    
    # 落在圆内的点
    inside = np.sum(distance <= 1)
    
    # π 的估计值
    # 圆面积 = π * r² = π * 1 = π
    # 正方形面积 = 4
    # π ≈ 4 * (inside / total)
    pi_estimate = 4 * inside / n_points
    
    return pi_estimate, x, y, distance <= 1

# 不同试验次数
trial_sizes = [100, 1000, 10000, 100000]

for n in trial_sizes:
    pi_est, _, _, _ = estimate_pi(n)
    error = abs(pi_est - np.pi)
    print(f"n = {n:6d}: π ≈ {pi_est:.6f}, 误差 = {error:.6f}")

In [ ]:
# 可视化
n_vis = 5000
pi_est, x, y, inside = estimate_pi(n_vis)

fig, ax = plt.subplots(figsize=(8, 8))

# 圆外的点（红色）
ax.scatter(x[~inside], y[~inside], c='red', s=1, alpha=0.5, label='圆外')
# 圆内的点（蓝色）
ax.scatter(x[inside], y[inside], c='blue', s=1, alpha=0.5, label='圆内')

# 画圆
theta = np.linspace(0, np.pi/2, 100)
ax.plot(np.cos(theta), np.sin(theta), 'g-', linewidth=2, label='单位圆')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.set_title(f'Monte Carlo 估计 π\nn={n_vis}, π ≈ {pi_est:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 生日问题

**问题**：一个房间需要多少人，才能使至少两人生日相同的概率 ≥ 50%？

In [ ]:
def birthday_simulation(n_people, n_simulations=10000):
    """模拟生日问题"""
    same_birthday_count = 0
    
    for _ in range(n_simulations):
        # 随机生成 n_people 的生日（假设 365 天）
        birthdays = np.random.randint(1, 366, size=n_people)
        
        # 检查是否有重复
        if len(birthdays) != len(set(birthdays)):
            same_birthday_count += 1
    
    return same_birthday_count / n_simulations

# 测试不同人数
people_counts = [10, 20, 23, 30, 40, 50]

print("人数\t模拟概率\t理论概率")
print("-" * 40)

for n in people_counts:
    # 理论概率：P(无重复) = 365/365 × 364/365 × ... × (365-n+1)/365
    prob_no_duplicate = np.prod(np.arange(365, 365-n, -1)) / (365**n)
    theoretical_prob = 1 - prob_no_duplicate
    
    simulated_prob = birthday_simulation(n)
    
    print(f"{n}\t{simulated_prob:.4f}\t\t{theoretical_prob:.4f}")

## 3. 大数定律

In [ ]:
# 大数定律演示：随着试验次数增加，样本均值收敛到期望值

def law_of_large_numbers_demo(dist_func, expected_value, max_trials=10000):
    """演示大数定律"""
    trials = np.arange(1, max_trials + 1)
    cumulative_mean = np.cumsum(dist_func(max_trials)) / trials
    
    return trials, cumulative_mean, expected_value

# 示例 1：掷骰子
trials, mean_1, ev_1 = law_of_large_numbers_demo(
    lambda n: np.random.randint(1, 7, n), 3.5
)

# 示例 2：抛硬币
trials, mean_2, ev_2 = law_of_large_numbers_demo(
    lambda n: np.random.randint(0, 2, n), 0.5
)

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(trials, mean_1, alpha=0.7, linewidth=0.5)
axes[0].axhline(ev_1, color='red', linestyle='--', label=f'期望值 = {ev_1}')
axes[0].set_xlabel('试验次数')
axes[0].set_ylabel('累积均值')
axes[0].set_title('大数定律：掷骰子')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(trials, mean_2, alpha=0.7, linewidth=0.5)
axes[1].axhline(ev_2, color='red', linestyle='--', label=f'期望值 = {ev_2}')
axes[1].set_xlabel('试验次数')
axes[1].set_ylabel('累积均值')
axes[1].set_title('大数定律：抛硬币')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 中心极限定理

In [ ]:
# 中心极限定理：大量独立随机变量之和趋向正态分布

def clt_demo(dist_func, n_samples, sample_size):
    """演示中心极限定理"""
    sample_means = []
    
    for _ in range(n_samples):
        sample = dist_func(sample_size)
        sample_means.append(np.mean(sample))
    
    return np.array(sample_means)

# 不同原始分布
distributions = [
    ('均匀分布', lambda n: np.random.uniform(0, 1, n)),
    ('指数分布', lambda n: np.random.exponential(1, n)),
    ('伯努利分布', lambda n: np.random.randint(0, 2, n)),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sample_size = 30
n_samples = 10000

for i, (name, dist_func) in enumerate(distributions):
    # 原始分布
    original_data = dist_func(10000)
    axes[0, i].hist(original_data, bins=50, density=True, alpha=0.7)
    axes[0, i].set_title(f'{name}\n(原始分布)')
    
    # 样本均值分布
    sample_means = clt_demo(dist_func, n_samples, sample_size)
    axes[1, i].hist(sample_means, bins=50, density=True, alpha=0.7)
    
    # 拟合正态分布
    mu, sigma = np.mean(sample_means), np.std(sample_means)
    x = np.linspace(min(sample_means), max(sample_means), 100)
    axes[1, i].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='正态拟合')
    axes[1, i].set_title(f'{name}\n样本均值分布 (n={sample_size})')
    axes[1, i].legend()

plt.tight_layout()
plt.show()

## 5. 实际应用：需求预测

In [ ]:
# 单车需求模拟
# 假设某站点每小时需求服从泊松分布，参数 λ = 20

def simulate_demand(lam, hours, n_simulations=1000):
    """模拟多天的需求"""
    all_demands = []
    
    for _ in range(n_simulations):
        daily_demand = np.random.poisson(lam, hours)
        all_demands.append(daily_demand)
    
    return np.array(all_demands)

# 模拟
lam = 20  # 平均每小时需求
hours = 24
n_sim = 10000

demands = simulate_demand(lam, hours, n_sim)

# 分析
daily_totals = demands.sum(axis=1)

print("单车需求模拟结果")
print("=" * 40)
print(f"平均每小时需求 λ = {lam}")
print(f"每日总需求均值: {np.mean(daily_totals):.2f}")
print(f"每日总需求标准差: {np.std(daily_totals):.2f}")
print(f"95% 置信区间: [{np.percentile(daily_totals, 2.5):.0f}, {np.percentile(daily_totals, 97.5):.0f}]")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 每小时需求分布
axes[0].hist(demands[:, 12], bins=30, density=True, alpha=0.7, label='模拟')
x = np.arange(0, 40)
axes[0].plot(x, stats.poisson.pmf(x, lam), 'ro-', label='理论值')
axes[0].set_xlabel('需求量')
axes[0].set_ylabel('概率密度')
axes[0].set_title('中午 12 点需求分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 每日总需求分布
axes[1].hist(daily_totals, bins=50, density=True, alpha=0.7, label='模拟')
x = np.linspace(min(daily_totals), max(daily_totals), 100)
mu, sigma = np.mean(daily_totals), np.std(daily_totals)
axes[1].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='正态拟合')
axes[1].set_xlabel('每日总需求')
axes[1].set_ylabel('概率密度')
axes[1].set_title('每日总需求分布 (CLT 验证)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Research Thinking

### 问题 1：模拟 vs 理论

什么情况下用模拟比用理论计算更好？

**回答：**

1. **复杂系统**：多个随机变量相互作用，理论推导困难
2. **无解析解**：某些概率分布没有闭式解
3. **快速验证**：先模拟验证直觉，再推导理论
4. **教学演示**：直观展示统计概念

### 问题 2：模拟精度

多少次模拟才够？如何确定模拟次数？

**回答：**

- 标准误差 ∝ 1/√n
- 要将误差减半，需要 4 倍的模拟次数
- 实践中：先小规模试验，观察收敛趋势，再决定最终次数
- 通常 10000-100000 次已足够精确

### 问题 3：随机种子

为什么设置随机种子？什么时候不该设置？

**回答：**

**应该设置：**
- 调试代码时
- 需要可重复结果时
- 教学演示时

**不该设置：**
- 最终生产环境
- 需要真正的随机性时
- 蒙特卡罗积分（不同种子得到不同估计）

## 7. 练习

### 练习 1
模拟 100 个人同时抛硬币，每人抛 10 次。计算恰好有 5 次正面的人数分布。

In [ ]:
# 你的代码


### 练习 2
使用 Monte Carlo 方法计算定积分 ∫₀¹ x² dx = 1/3

In [ ]:
# 你的代码


### 练习 3
某站点需求服从 λ=15 的泊松分布。如果站点有 20 辆车，求需求超过供给的概率。

In [ ]:
# 你的代码


## 8. 总结

### 本周学习要点

1. **概率基础**：样本空间、事件、概率分布
2. **Monte Carlo 模拟**：用随机试验估计确定性量
3. **大数定律**：样本均值收敛到期望
4. **中心极限定理**：样本均值分布趋向正态
5. **实际应用**：需求预测、风险评估

### 关键洞察

- 模拟次数越多，结果越精确
- CLT 使得我们可用正态分布近似样本均值分布
- 设置随机种子保证可重复性